In [300]:
import sys
import os
# Get the current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all from the-machine and custom_matroid_functions
from HM_module import *
from custom_matroid_functions import *

In [301]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()
k, n = 4, 6
#M = matroids.CompleteGraphic(n)
M = matroids.Uniform(k, n)
#M = matroids.catalog.Fano()
L = M.lattice_of_flats()
r = M.rank()
rank_func = lambda x : M.rank(x)

#e = next(x for x in M.groundset() if x  not in M.coloops())
e = list(M.groundset())[0]
print(M.groundset())
print("deleting ", e)
mdele = M.delete(e)
Eq_M = create_deformed_algebra(M, base_ring=P)
Eq_mdele = create_deformed_algebra(mdele, base_ring=P)
E_mdele = Eq_mdele.natural()

E = Eq_M.natural()
E_basis = E.basis()
E_mdele_basis = E_mdele.basis()

C = Eq_M.C()
C_dele = Eq_mdele.C()
C_basis = C.basis()
C_dele_basis = C_dele.basis()

flats_keys = list(E_basis.keys())

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
char_poly = characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
grev = lambda M: P(x**M.rank())
mu_rev = lambda M: P(x**M.rank() * chibar(M)(0))
mu = lambda M: P(char_poly(M)(0))

print(chibar(M))
print(mu_rev(M))
print(kl_poly(M))

frozenset({0, 1, 2, 3, 4, 5})
deleting  0
x^3 - 5*x^2 + 10*x - 10
-10*x^4
1 + 14*x


In [302]:
def flat_division(M, e):
    L = M.lattice_of_flats()
    flat_div = {'S':set(), 'T':set(), 'A':set(), 'B':set()}
    for F in L:
        if e in F:
            if F - {e} in L:
                flat_div['T'].add(F)
            else:
                flat_div['A'].add(F)
        else:
            if F.union({e}) in L:
                flat_div['S'].add(F)
            else:
                flat_div['B'].add(F)
    return flat_div

def get_pal0_part(f):
    """
    Extracts the Pal(0) part of a Laurent polynomial f.
    """
    if f == 0: return f
    x_var = f.parent().gen()
    f_inv = f.subs({x_var: x_var**-1})
    numerator = (x_var**2) * f_inv - f
    denominator = (x_var**2) - 1
    q, r = numerator.quo_rem(denominator)
    return q


flat_div = flat_division(M, e)
#flat_div['S'].remove(frozenset())

def convolve(f, g):
    return lambda M: sum([f(M.delete(M.groundset() - F))(x) * g(M.contract(F))(x) for F in list(M.lattice_of_flats())])

conv = lambda f, g: convolve(f, g)(M)

def rev(f):
    return lambda M: f(M)(x**-1) * x**M.rank()

rightAugpoly = convolve(chow, rev(kl_poly))
bbasepoly = convolve(mu_rev, rightAugpoly)
zeta = lambda M: P(1) if M else 0
klrev = rev(kl_poly)
leftAugpoly = convolve(grev, chow_polynomial)

In [303]:
print("rank of M: ", r)
print("chow polynomial of M: ", chow_polynomial(M))
print(reduced_characteristic_polynomial(M))
for F in flat_div['S']:
    print(F)

rank of M:  4
chow polynomial of M:  x^3 + 36*x^2 + 36*x + 1
x^3 - 5*x^2 + 10*x - 10
frozenset({3, 4})
frozenset({2})
frozenset({1, 4})
frozenset({2, 3})
frozenset({1, 2})
frozenset({4, 5})
frozenset({2, 5})
frozenset({3})
frozenset({2, 4})
frozenset({1})
frozenset({1, 5})
frozenset({3, 5})
frozenset()
frozenset({1, 3})
frozenset({4})
frozenset({5})


In [304]:
print(bbasepoly(M))


-1 - 50*x - 50*x^2 + 15*x^3


In [305]:
def deleconj(M, e, f):
    mdele = M.delete(e)
    og_matroid_poly = f(M)
    deleted_matroid_poly = f(mdele)
    con_matroid_poly = f(M.contract(e))
    sum2 = 0
    GinvF = convolve(convolve(chibar, mu_rev), f)
    for F in flat_div['S']:
        poly1 = chow_polynomial(M.contract(F.union({e})))
        poly2 = f(M.delete(M.groundset() - F))
        term = x * poly1 * poly2
        sum2 += term
        print(F)
        print("poly1:   ", poly1)
        print("poly2:   ", poly2)
        print("term: ", term)
        print("--------------------------------")
        print("********************************")
    print("summation equals:", sum2)
    print("f(M):     ", og_matroid_poly)
    print("f(M\\e):  ", deleted_matroid_poly)
    print("f(M/e):  ", con_matroid_poly)
    return P(deleted_matroid_poly + sum2)

def deleconjp2(M, e, f):
    sum2 = 0
    GinvF = convolve(convolve(chibar, mu_rev), rightAugpoly)
    for F in flat_div['T']:
        poly1 = bbasepoly(M.contract(F))
        poly2 = chow_polynomial(M.delete(M.groundset()))
        term = poly1 * poly2
        sum2 += term
    return P(sum2)

right = deleconj(M, e, rightAugpoly)
print("right", right)

frozenset({3, 4})
poly1:    1
poly2:    1 + 3*x + x^2
term:  x + 3*x^2 + x^3
--------------------------------
********************************
frozenset({2})
poly1:    x + 1
poly2:    1 + x
term:  x + 2*x^2 + x^3
--------------------------------
********************************
frozenset({1, 4})
poly1:    1
poly2:    1 + 3*x + x^2
term:  x + 3*x^2 + x^3
--------------------------------
********************************
frozenset({2, 3})
poly1:    1
poly2:    1 + 3*x + x^2
term:  x + 3*x^2 + x^3
--------------------------------
********************************
frozenset({1, 2})
poly1:    1
poly2:    1 + 3*x + x^2
term:  x + 3*x^2 + x^3
--------------------------------
********************************
frozenset({4, 5})
poly1:    1
poly2:    1 + 3*x + x^2
term:  x + 3*x^2 + x^3
--------------------------------
********************************
frozenset({2, 5})
poly1:    1
poly2:    1 + 3*x + x^2
term:  x + 3*x^2 + x^3
--------------------------------
********************************
frozen

In [306]:
print(deleconjp2(M, e, rightAugpoly))

-16 - 21*x - 6*x^2


In [307]:
print(rightAugpoly(M))
print(leftAugpoly(M))

1 + 56*x + 161*x^2 + 56*x^3 + x^4
1 + 42*x + 117*x^2 + 42*x^3 + x^4
